In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
import random
from PIL import Image
import timm
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import random

In [2]:
def set_random_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = "42"
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

set_random_seed()

In [3]:
df = pd.read_csv('/kaggle/input/peta-train-test/PETA_train_list.txt', sep=" ", header=None).iloc[:,:36]
df[0] = '/kaggle/input/peta-v1/PETA dataset/'+df[0].str.split('/').str[1:].str.join("/")

train_set, val_set = train_test_split(df, test_size=0.1666, random_state=42, shuffle=True)

test_set = pd.read_csv('/kaggle/input/peta-train-test/PETA_test_list.txt', sep=" ", header=None).iloc[:,:36]
test_set[0] = '/kaggle/input/peta-v1/PETA dataset/'+test_set[0].str.split('/').str[1:].str.join("/")

In [4]:
class CustomImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.imgs = df.iloc[:,0].values
        self.labels = df.iloc[:,1:].values.astype('float32')
        self.transform = transform
        
    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, idx):
        image = Image.open(self.imgs[idx]).convert('RGB')
        label = torch.from_numpy(self.labels[idx])
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # maintain pedestrian shape, taller than wide
    transforms.RandomHorizontalFlip(),  # common for pedestrian symmetry
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor()
])

train_dataset = CustomImageDataset(train_set, train_transform)
val_dataset = CustomImageDataset(val_set, val_transform)
test_dataset = CustomImageDataset(test_set, val_transform)

In [6]:
r = train_set.iloc[:,1:].values.mean(0)
r = torch.from_numpy(r).type(torch.float32)

In [7]:
# Common DataLoader kwargs
loader_kwargs = {
    'batch_size': 16,
    'num_workers': 4,
    'pin_memory': True,
    'persistent_workers': True,
}

train_loader = DataLoader(
    train_dataset, 
    shuffle=True,
    **loader_kwargs
)

val_loader = DataLoader(
    val_dataset, 
    shuffle=False,
    **loader_kwargs
)

test_loader = DataLoader(
    test_dataset, 
    shuffle=False,
    **loader_kwargs
)

In [8]:
def compute_mFive(y_true, y_pred):
    y_true = np.array(y_true, dtype='float32')
    y_pred = np.array(y_pred, dtype='float32')
    
    # ---- mA (mean per-label accuracy) ----
    L = y_true.shape[1]
    TP = np.sum((y_true == 1) & (y_pred == 1), axis=0)
    TN = np.sum((y_true == 0) & (y_pred == 0), axis=0)
    P = np.sum(y_true == 1, axis=0)
    N = np.sum(y_true == 0, axis=0)

    tp_over_p = np.where(P == 0, 1.0, TP / P)
    tn_over_n = np.where(N == 0, 1.0, TN / N)

    mA = (1 / (2 * L)) * np.sum(tp_over_p + tn_over_n)

    # ---- Instance-level metrics ----
    intersection = np.logical_and(y_true, y_pred).sum(axis=1)
    union = np.logical_or(y_true, y_pred).sum(axis=1)
    pred_sum = y_pred.sum(axis=1)
    true_sum = y_true.sum(axis=1)

    eps = 1e-8  # small number to avoid division by zero
    Accuracy = np.mean(intersection / (union + eps))
    Precision = np.mean(intersection / (pred_sum + eps))
    Recall = np.mean(intersection / (true_sum + eps))
    F1 = 2 * Precision * Recall / (Precision + Recall)

    # ---- mFive ----
    mFive = np.mean([mA, Accuracy, Precision, Recall, F1])

    return {
        'mA': mA,
        'Accuracy': Accuracy,
        'Precision': Precision,
        'Recall': Recall,
        'F1': F1,
        'mFive': mFive
    }

In [9]:
def train_one_epoch(model, dataloader, criterion, optimizer, scheduler, device, epoch, num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} - Training", leave=False)
    for i, (images, labels) in enumerate(pbar, 1):
        images = images.to(device)
        labels = torch.abs(labels-torch.rand(labels.shape) * 0.2)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        labels = (labels >= 0.5).float()
        correct += (preds == labels).float().sum().item()
        total += labels.numel()

        pbar.set_postfix({
            'Avg Loss': f'{running_loss / i:.4f}',
            'Avg Acc': f'{correct / total:.4f}'
        })
        
    scheduler.step()
    return running_loss / len(dataloader), correct / total


def validate_one_epoch(model, dataloader, criterion, device, epoch, num_epochs):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation", leave=False)
    with torch.no_grad():
        for i, (images, labels) in enumerate(pbar, 1):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).float().sum().item()
            total += labels.numel()

            all_predictions.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({
                'Avg Loss': f'{running_loss / i:.4f}',
                'Avg Acc': f'{correct / total:.4f}'
            })

    metrics = compute_mFive(all_labels, all_predictions)
    return running_loss / len(dataloader), correct / total, metrics


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=20):
    model.to(device)
    best_val_mfive = 0
    train_losses, val_losses = [], []

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device, epoch, num_epochs)
        val_loss, val_acc, metrics = validate_one_epoch(model, val_loader, criterion, device, epoch, num_epochs)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"\nEpoch {epoch+1}/{num_epochs} Summary:")
        print(f"  Train Loss:     {train_loss:.4f} - Acc: {train_acc:.4f}")
        print(f"  Val Loss:{val_loss:.4f} - Val Acc: {val_acc:.4f}")
        print(f"  Val mFive:      {metrics['mFive']:.4f}")
        print(f"  Val Acc:         {metrics['Accuracy']:.4f}")
        print(f"  Val mA:         {metrics['mA']:.4f}")
        print(f"  Val F1:         {metrics['F1']:.4f}")
        print(f"  Val Precision:  {metrics['Precision']:.4f}")
        print(f"  Val Recall:     {metrics['Recall']:.4f}")
        print("-" * 60)

        if metrics["mFive"] > best_val_mfive:
            best_val_mfive = metrics["mFive"]
            torch.save(model.state_dict(), 'best_weight.pth')
            print(f"✅ Best model saved (mFive: {best_val_mfive:.4f})")

    return model, train_losses, val_losses

def test_model(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    pbar = tqdm(test_loader, desc="Testing", leave=False)
    with torch.no_grad():
        for i, (images, labels) in enumerate(pbar, 1):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).float().sum().item()
            total += labels.numel()

            all_predictions.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            avg_loss = running_loss / i
            avg_acc = correct / total

            pbar.set_postfix({
                'Avg Loss': f'{avg_loss:.4f}',
                'Avg Acc': f'{avg_acc:.4f}'
            })

    metrics = compute_mFive(all_labels, all_predictions)

    print(f"\nTest Summary:")
    print(f"  Test Loss:     {running_loss / len(test_loader):.4f}")
    print(f"  Test mFive:    {metrics['mFive']:.4f}")
    print(f"  Test Accuracy: {metrics['Accuracy']:.4f}")
    print(f"  Test mA:       {metrics['mA']:.4f}")
    print(f"  Test F1:       {metrics['F1']:.4f}")
    print(f"  Test Precision:{metrics['Precision']:.4f}")
    print(f"  Test Recall:   {metrics['Recall']:.4f}")
    print("-" * 60)

    return metrics

In [11]:
class Swin_Baseline(nn.Module):
    def __init__(self, num_classes):
        super(Swin_Baseline, self).__init__()
        self.backbone = timm.create_model("swin_small_patch4_window7_224", pretrained=True, features_only=True)
        self.pool = nn.AdaptiveAvgPool2d(1) 
        self.classifier = nn.Linear(768, num_classes)  

    def forward(self, x):
        x = self.backbone(x)[-1].permute(0,3, 1, 2)        
        x = self.pool(x)              
        x = torch.flatten(x, 1)       
        x = self.classifier(x)        
        return x

In [12]:
class WeightedBCELoss(nn.Module):
    
    def __init__(self, r):
        super(WeightedBCELoss, self).__init__()
        self.r = r 

    def forward(self, logits, targets): 
        probs = torch.sigmoid(logits) 
        pos_weights = torch.exp(1 - self.r).expand_as(targets)
        neg_weights = torch.exp(self.r).expand_as(targets)
        weighted_bce = pos_weights * targets * torch.log(probs + 1e-7) + neg_weights * (1 - targets) * torch.log(1 - probs + 1e-7)
        return (-weighted_bce).mean()


In [13]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model=Swin_Baseline(num_classes=35)
# Load checkpoint
checkpoint = torch.load('/kaggle/input/sbline/best_weight.pth', map_location=device)
# Remove only head weights
state_dict = {k: v for k, v in checkpoint.items() if not (k.startswith("classifier") or k.startswith("classifier.lam"))}
model.load_state_dict(state_dict, strict=False)

model.safetensors:   0%|          | 0.00/200M [00:00<?, ?B/s]

_IncompatibleKeys(missing_keys=['classifier.weight', 'classifier.bias'], unexpected_keys=[])

In [14]:
criterion = WeightedBCELoss(r.to(device)).to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.001)
    
# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=11, gamma=0.1)

In [15]:
model, train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=50)


Epoch 1/50 Summary:
  Train Loss:     0.8235 - Acc: 0.8596
  Val Loss:0.5073 - Val Acc: 0.8939
  Val mFive:      0.7700
  Val Acc:         0.6744
  Val mA:         0.8005
  Val F1:         0.7912
  Val Precision:  0.7651
  Val Recall:     0.8191
------------------------------------------------------------
✅ Best model saved (mFive: 0.7700)



Epoch 2/50 Summary:
  Train Loss:     0.7401 - Acc: 0.9157
  Val Loss:0.4530 - Val Acc: 0.9185
  Val mFive:      0.8167
  Val Acc:         0.7437
  Val mA:         0.8301
  Val F1:         0.8363
  Val Precision:  0.8202
  Val Recall:     0.8531
------------------------------------------------------------
✅ Best model saved (mFive: 0.8167)



Epoch 3/50 Summary:
  Train Loss:     0.7048 - Acc: 0.9381
  Val Loss:0.4342 - Val Acc: 0.9225
  Val mFive:      0.8293
  Val Acc:         0.7593
  Val mA:         0.8467
  Val F1:         0.8466
  Val Precision:  0.8293
  Val Recall:     0.8648
------------------------------------------------------------
✅ Best model saved (mFive: 0.8293)



Epoch 4/50 Summary:
  Train Loss:     0.6825 - Acc: 0.9526
  Val Loss:0.4170 - Val Acc: 0.9280
  Val mFive:      0.8397
  Val Acc:         0.7742
  Val mA:         0.8561
  Val F1:         0.8559
  Val Precision:  0.8425
  Val Recall:     0.8698
------------------------------------------------------------
✅ Best model saved (mFive: 0.8397)



Epoch 5/50 Summary:
  Train Loss:     0.6651 - Acc: 0.9631
  Val Loss:0.4153 - Val Acc: 0.9292
  Val mFive:      0.8439
  Val Acc:         0.7783
  Val mA:         0.8641
  Val F1:         0.8589
  Val Precision:  0.8460
  Val Recall:     0.8722
------------------------------------------------------------
✅ Best model saved (mFive: 0.8439)


...


Epoch 46/50 Summary:
  Train Loss:     0.5969 - Acc: 0.9991
  Val Loss:0.3950 - Val Acc: 0.9382
  Val mFive:      0.8602
  Val Acc:         0.8031
  Val mA:         0.8718
  Val F1:         0.8754
  Val Precision:  0.8664
  Val Recall:     0.8845
------------------------------------------------------------



Epoch 47/50 Summary:
  Train Loss:     0.5967 - Acc: 0.9991
  Val Loss:0.3950 - Val Acc: 0.9382
  Val mFive:      0.8603
  Val Acc:         0.8032
  Val mA:         0.8718
  Val F1:         0.8754
  Val Precision:  0.8665
  Val Recall:     0.8845
------------------------------------------------------------



Epoch 48/50 Summary:
  Train Loss:     0.5962 - Acc: 0.9991
  Val Loss:0.3950 - Val Acc: 0.9383
  Val mFive:      0.8603
  Val Acc:         0.8033
  Val mA:         0.8719
  Val F1:         0.8755
  Val Precision:  0.8666
  Val Recall:     0.8845
------------------------------------------------------------



Epoch 49/50 Summary:
  Train Loss:     0.5960 - Acc: 0.9991
  Val Loss:0.3950 - Val Acc: 0.9382
  Val mFive:      0.8603
  Val Acc:         0.8032
  Val mA:         0.8718
  Val F1:         0.8754
  Val Precision:  0.8665
  Val Recall:     0.8845
------------------------------------------------------------



Epoch 50/50 Summary:
  Train Loss:     0.5964 - Acc: 0.9990
  Val Loss:0.3950 - Val Acc: 0.9382
  Val mFive:      0.8603
  Val Acc:         0.8032
  Val mA:         0.8718
  Val F1:         0.8754
  Val Precision:  0.8665
  Val Recall:     0.8845
------------------------------------------------------------


In [16]:
model.load_state_dict(torch.load('best_weight.pth'))
metrics = test_model(model, test_loader, criterion, device)


Test Summary:
  Test Loss:     0.3875
  Test mFive:    0.8656
  Test Accuracy: 0.8099
  Test mA:       0.8782
  Test F1:       0.8799
  Test Precision:0.8724
  Test Recall:   0.8876
------------------------------------------------------------
